In [1]:
%load_ext autoreload
%autoreload 2
import sys; sys.path.insert(0, "..")

In [2]:

from evaluate import load_benchmark
recs = load_benchmark("ikp")
print(len(recs)); recs[0]


268351


{'subject_id': 'd6ec2555f79d0d80',
 'item_id': 'b4e25751c476d271',
 'subject': {'normalized_name': 'Anthropic Claude 3 Haiku',
  'provider': 'Anthropic',
  'release_date': '2024-03-04',
  'access_date': '',
  'harness': '',
  'harness_version': '',
  'reasoning_effort': '',
  'subject_features_extra': ''},
 'item': {'item_content': 'What strait separates Sicily from mainland Italy?',
  'item_features': '',
  'benchmark_id': 'bench_bf8b9fff257c',
  'interactors': ''},
 'label': 1}

In [3]:
from evaluate import split_pairs, index_responses, labeled_at
pairs = split_pairs(recs); idx = index_responses(recs)
acq, ev = pairs["d6ec2555f79d0d80"]
print(len(pairs), "subjects qualify |", len(acq), "acquisition items,", len(ev), "evaluation items for Claude 3 Haiku")
print("overlap:", len(set(acq) & set(ev)))
for b in [0, 1, 3, 7]:
    print(f"budget {b}: {len(labeled_at(pairs, idx, b))} reveals pooled")


170 subjects qualify | 640 acquisition items, 641 evaluation items for Claude 3 Haiku
overlap: 0
budget 0: 0 reveals pooled
budget 1: 170 reveals pooled
budget 3: 510 reveals pooled
budget 7: 1190 reveals pooled


In [4]:
from evaluate import evaluate_benchmark, alc
scores = evaluate_benchmark(lambda inp, labeled: 0.5, recs, max_eval_items=20)
s = scores["d6ec2555f79d0d80"]
print(s, "\nALC:", alc(s))


{0: 0.25, 1: 0.25, 3: 0.25, 7: 0.25, 15: 0.25, 31: 0.25} 
ALC: 0.24999999999999997


In [5]:
from evaluate import evaluate
evaluate(lambda inp, labeled: 0.5, ["ikp", "tau2_bench"], max_eval_items=20)


  ikp                          pairs= 170  ALC=0.2500
  tau2_bench                   pairs=  29  ALC=0.2500
  B0=0.250  B1=0.250  B3=0.250  B7=0.250  B15=0.250  B31=0.250
  Brier ALC = 0.2500 over 199 pairs


{'alc': 0.24999999999999997,
 'per_budget': {0: 0.25, 1: 0.25, 3: 0.25, 7: 0.25, 15: 0.25, 31: 0.25},
 'per_benchmark': {'ikp': 0.24999999999999994,
  'tau2_bench': 0.24999999999999997},
 'n_pairs': 199}

In [6]:
from evaluate import EmpiricalMean
evaluate(EmpiricalMean(), ["ikp", "tau2_bench"], max_eval_items=20)


  ikp                          pairs= 170  ALC=0.2913
  tau2_bench                   pairs=  29  ALC=0.2230
  B0=0.250  B1=0.421  B3=0.279  B7=0.245  B15=0.227  B31=0.219
  Brier ALC = 0.2814 over 199 pairs


{'alc': 0.2813516785773193,
 'per_budget': {0: 0.25,
  1: 0.42093704531320697,
  3: 0.27915399255576157,
  7: 0.24473876293505276,
  15: 0.22721222757035367,
  31: 0.21943272902444394},
 'per_benchmark': {'ikp': 0.2913093147917047,
  'tau2_bench': 0.22297932835506112},
 'n_pairs': 199}

In [7]:
from evaluate import ShrunkMean
evaluate(ShrunkMean(1, 1), ["ikp", "tau2_bench"], max_eval_items=20)


  ikp                          pairs= 170  ALC=0.2438
  tau2_bench                   pairs=  29  ALC=0.1972
  B0=0.250  B1=0.251  B3=0.242  B7=0.233  B15=0.224  B31=0.219
  Brier ALC = 0.2370 over 199 pairs


{'alc': 0.23701055827312625,
 'per_budget': {0: 0.25,
  1: 0.2514234595488466,
  3: 0.24179725315489747,
  7: 0.233221035480908,
  15: 0.22427856389312673,
  31: 0.2186649585757047},
 'per_benchmark': {'ikp': 0.24379781970307432,
  'tau2_bench': 0.19722316368377554},
 'n_pairs': 199}

In [8]:
from evaluate import HierarchicalMean
evaluate(HierarchicalMean(), ["ikp", "tau2_bench"], max_eval_items=20)


  ikp                          pairs= 170  ALC=0.2358
  tau2_bench                   pairs=  29  ALC=0.1847
  B0=0.250  B1=0.234  B3=0.229  B7=0.225  B15=0.221  B31=0.217
  Brier ALC = 0.2284 over 199 pairs


{'alc': 0.22839372798043042,
 'per_budget': {0: 0.25,
  1: 0.23374553551919522,
  3: 0.22901999609433019,
  7: 0.22472207536237845,
  15: 0.22073745787941867,
  31: 0.21748715009365813},
 'per_benchmark': {'ikp': 0.2358473498782546,
  'tau2_bench': 0.18470008237249577},
 'n_pairs': 199}

#### Embedding tests

In [10]:
sys.path.insert(0, "../submission")
from model import subject_name, embed
print(subject_name(load_benchmark("alpha_sql")[0]["subject"]), "|", subject_name(recs[0]["subject"]))
v = embed(["What strait separates Sicily from mainland Italy?", "Which strait lies between Sicily and Italy?", "Write a SQL query"])
print(v.shape, "| similar questions:", round(float(v[0] @ v[1]), 2), "| unrelated:", round(float(v[0] @ v[2]), 2))


Qwen/Qwen2.5-Coder-32B-Instruct | Anthropic Claude 3 Haiku


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

(3, 384) | similar questions: 0.92 | unrelated: 0.05


### checking NPZ files

In [13]:
import numpy as np
z = np.load("../data/feat/ikp.npz")
{k: z[k].shape for k in z.files}


{'item_emb': (1400, 384),
 'item_len': (1400,),
 'subj_emb': (170, 384),
 'subj_json': (170,),
 'subj_idx': (226211,),
 'item_idx': (226211,),
 'y': (226211,)}

## TESTING HAIKU TEXT RATING 

In [16]:
from model import rater_client, rate_text
rate_text(rater_client(), "What strait separates Sicily from mainland Italy?")


{'p_correct': 97.0, 'difficulty': 1.0, 'domain': 'knowledge'}